In [4]:
import mediapy as media
import numpy as np
import sys
import jax
import mujoco
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from pathlib import Path
from mujoco import mjx
REPO = Path.cwd().parent          # training/ -> robotic-craftsman/
sys.path.insert(0, str(REPO))
from arm import ROOT, NPZ_PATH, Calibration, HandPose, MjcArm, DepthScale, ArmIK, STServo, FingerBoard

In [5]:
CLIPS = sorted((ROOT / "refs").glob("*.npz"))
Q_REF = np.load(CLIPS[0])["q_ref"]

In [6]:
sim = MjcArm()
POLICY_ACTS = ["a_elbow_pitch", "a_forearm_roll", "a_wrist_tilt", "a_index_mcp", "a_thumb_mcp", "a_thumb_pip"]
idx_list = []
for name in POLICY_ACTS:
    idx_list.append(sim.actuator_id(name))

LOWERS, UPPERS = sim.model.actuator_ctrlrange[idx_list].T
print(LOWERS, UPPERS)

[-1.57 -3.14 -1.4  -0.05 -1.27 -0.05] [1.57 3.14 1.4  1.2  0.1  1.25]


In [7]:
#smoothing
Q_REF = np.clip(Q_REF, LOWERS, UPPERS)
Q_REF = DepthScale.smooth(Q_REF)
print(Q_REF.shape)
elbow = Q_REF[:, 0]
wrist_roll = Q_REF[:, 1]
wrist_tilt = Q_REF[:, 2]
index = Q_REF[:,3]
thumb_add = Q_REF[:,4]
thumb_flex = Q_REF[:,5]

(121, 6)


In [ ]:
#plotting
plt.plot([joint for Q_REF[:, joint]], [t for Q_REF[t]])
plt.ylabel('joint value')
plt.show()

In [8]:
mx = mjx.put_model(sim.model)
dx = mjx.put_data(sim.model, sim.data)
dx = mjx.step(mx, dx)

mjx.get_data(sim.model, dx)
sid = mujoco.mj_name2id(sim.model, mujoco.mjtObj.mjOBJ_SENSOR, "index_touch")
adr = sim.model.sensor_adr[sid]
dx.sensordata[adr]

step = jax.jit(mjx.step)       # mesh centre vs body origin
adr = sim.model.jnt_qposadr[sim.model.joint("needle_free").id]
sim.data.qpos[adr:adr + 3] = [1,1,1]

sim.set_ctrl("a_index_mcp",0.55)
sim.set_ctrl("a_thumb_pip", -0.05)
sim.set_ctrl("a_thumb_mcp", -0.0233)

mx = mjx.put_model(sim.model)
dx = mjx.put_data(sim.model, sim.data)

for _ in range (1500):
    dx = step(mx, dx)

if np.isnan(dx.qpos).any():
    raise ValueError("NaN in qpos")

sid = mujoco.mj_name2id(sim.model, mujoco.mjtObj.mjOBJ_SENSOR, "index_touch")
sth = mujoco.mj_name2id(sim.model, mujoco.mjtObj.mjOBJ_SENSOR, "thumb_touch")
adr_i = sim.model.sensor_adr[sid]
adr_t = sim.model.sensor_adr[sth]
print(dx.sensordata[adr_i], dx.sensordata[adr_t])

42.66696 42.66696


In [ ]:
biggest_jump = []
for i in range(Q_REF.shape[1]):
    fr_diff = []
    fr = 0
    for fr in range (len(Q_REF[:, i]) - 1):
        diff = np.abs(Q_REF[fr, i] - Q_REF[fr+1, i])
        fr_diff.append(diff)
        fr += 1
    biggest_jump.append(max(fr_diff))
    fr_diff = []
ACTION_SCALE = np.array(biggest_jump) * 1.5
print(dict(zip(POLICY_ACTS, ACTION_SCALE.round(3))))


{'a_elbow_pitch': np.float64(0.113), 'a_forearm_roll': np.float64(1.427), 'a_wrist_tilt': np.float64(0.289), 'a_index_mcp': np.float64(0.0), 'a_thumb_mcp': np.float64(0.004), 'a_thumb_pip': np.float64(0.002)}


In [ ]:
CTRL_DT = 1/30
SIM_DT = 0.002 #timestep in sim
episode_length = 260
EMA_ALPHA = 0.8 #exponential moving average  - smoothing filter
noise = 0.01
tracking_sigma = 0.3 #eval of the copying score
max_tracking_error = 0.8 # maximum error allowance for target to arrive at
REWARD_WEIGHTS = {
    "tracking": 1.0,
    "pinch": 0.0,
    "action_rate": -0.01,
    "termination": -5.0
}